# gerbil-data: Feature Engineering Pipeline for Recommender Systems

End-to-end demo using the MovieLens 1M dataset.

In [ ]:
import os, json, pandas as pd
DATA_DIR = "/tmp/ml-1m"
OUTPUT_DIR = "/tmp/gerbil-output"
JAR = "target/gerbil-data-1.0.0-jar-with-dependencies.jar"
assert os.path.exists(DATA_DIR), f"Dataset not found at {DATA_DIR}"
print(f"Data: {DATA_DIR}")

## 1. Raw Data

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.dat"), sep="::",
    names=["UserID","MovieID","Rating","Timestamp"], engine="python", encoding="latin-1")
print(f"Ratings: {ratings.shape[0]} rows")
ratings.head()

In [ ]:
users = pd.read_csv(os.path.join(DATA_DIR, "users.dat"), sep="::",
    names=["UserID","Gender","Age","Occupation","ZipCode"], engine="python", encoding="latin-1")
print(f"Users: {users.shape[0]} rows")
users.head()

In [ ]:
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.dat"), sep="::",
    names=["MovieID","Title","Genres"], engine="python", encoding="latin-1")
print(f"Movies: {movies.shape[0]} rows")
movies.head()

## 2. ETL Pipeline (requires Spark)

In [ ]:
%%bash -s "$DATA_DIR" "$JAR"
DATA=$1; JAR=$2
spark-submit --class processing.clean.ML1MCleanSample $JAR $DATA
spark-submit --class processing.feature.ML1MMovieStatFeature $JAR $DATA
spark-submit --class processing.feature.ML1MUserMovieRateSequence $JAR $DATA
spark-submit --class processing.join.ML1MJoinSample $JAR $DATA
echo "ETL done"

## 3. Joined Feature Table

In [ ]:
join_path = os.path.join(DATA_DIR, "join_sample")
if os.path.exists(join_path):
    csv_files = [f for f in os.listdir(join_path) if f.endswith('.csv') and not f.startswith('.')]
    if csv_files:
        part = sorted(csv_files)[0]
        df = pd.read_csv(os.path.join(join_path, part), sep='\t', nrows=3,
            names=["uid","iid","ts","rating","day","user_prof","item_feat","user_beh"])
        for idx in range(len(df)):
            row = df.iloc[idx]
            try:
                up = json.loads(str(row["user_prof"]))
                print(f"Row {idx} user profile: {json.dumps(up, indent=2)}")
                break
            except:
                pass
else:
    print("Join sample not found. Run ETL first.")

## 4. Featurization (requires Spark)

In [ ]:
%%bash -s "$DATA_DIR" "$OUTPUT_DIR" "$JAR"
DATA=$1; OUT=$2; JAR=$3
spark-submit --class pipeline.ML1MPipeline --conf spark.serializer=org.apache.spark.serializer.JavaSerializer \
  $JAR --yesterday 20010101 --parts 1 --feature_threshold 5 --target_threshold 5 \
  --sample_ratio 0.1 --input_dir $DATA --output_dir $OUT --output_format tfrecord
echo "Featurization done"

## 5. TFRecord Output

In [ ]:
try:
    import tensorflow as tf
    tf_dir = os.path.join(OUTPUT_DIR, "20010101", "tfrecord")
    if os.path.exists(tf_dir):
        files = [f for f in os.listdir(tf_dir) if f.endswith(".tfrecord")]
        if files:
            ds = tf.data.TFRecordDataset(os.path.join(tf_dir, files[0]))
            for i, raw in enumerate(ds.take(1)):
                ex = tf.train.Example()
                ex.ParseFromString(raw.numpy())
                print(f"Fields: {len(ex.features.feature)}")
                for name in sorted(ex.features.feature)[:8]:
                    print(f"  {name}")
except ImportError:
    print("TensorFlow not installed (pip install tensorflow)")

## 6. Vocabulary

In [ ]:
vocab_path = os.path.join(OUTPUT_DIR, "20010101", "pos_map.json")
if os.path.exists(vocab_path):
    with open(vocab_path) as f:
        v = json.load(f)
    print(f"Vocabulary entries: {len(v)}")
else:
    print("Vocabulary not found.")

## Summary

1. Raw data inspection
2. ETL: raw data -> joined feature table
3. Featurization: features -> `{name}_raw/_index/_value` triplets
4. TFRecord output -> ready for TensorFlow training
5. Vocabulary -> embedding position maps for offline/online serving